# End-to-end EDA & Data Preprocessing: IMDB Top 1000 Movies

This notebook is designed for **beginners**. We will work with the **IMDB Top 1000 Movies & TV Shows** dataset,
where the goal is to analyze movies and predict **IMDB Rating** based on features like
genre, director, cast, runtime, certificate, and more.

We will cover:
- Loading and understanding the dataset
- Exploratory Data Analysis (EDA)
- Data cleaning & preprocessing

For each step we explain:
- **What** is being done
- **Why** we do it
- **What we learn / expect to find**

> Save your dataset CSV as `imdb_top_1000.csv` in the same directory as this notebook, or modify the path in the loading cell.


## Step 0 – Import libraries

**What:** Import the Python libraries used for data analysis.

**Why:** These provide ready-made tools for data tables, math, and visualizations.

- `pandas`: data manipulation (DataFrames)
- `numpy`: numerical operations
- `matplotlib` & `seaborn`: plotting and visualizations
- `sklearn`: tools for preprocessing and splitting into train/test


In [ ]:
# Step 0: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

%matplotlib inline
sns.set(style='whitegrid')


## Step 1 – Load the dataset

**What:** Read the CSV file into a `pandas` DataFrame.

**Why:** A DataFrame is the main structure used for EDA and preprocessing.

**What we learn:**
- Whether the file loads correctly
- How the first few rows of the data look


In [ ]:
# Step 1: Load the dataset
df = pd.read_csv('imdb_top_1000.csv')

df.head(10)


In [ ]:
df.shape  # Rows x Columns


## Step 2 – Column descriptions

Here is a clear description of each column in the IMDB Top 1000 Movies dataset:

1. **Poster_Link**
   * **Type:** String (URL)
   * **Meaning:** Link to the movie's poster image hosted on IMDB.
   * **Note:** Not useful for ML modeling — will be dropped during preprocessing.

2. **Series_Title**
   * **Type:** Categorical (string)
   * **Meaning:** Official title / name of the movie or TV show.
   * **Note:** Acts as a unique identifier per row — not useful as a feature.

3. **Released_Year**
   * **Type:** String → Numeric (after cleaning)
   * **Meaning:** The calendar year in which the movie was released.
   * **Note:** Stored as string; needs `pd.to_numeric()` conversion.

4. **Certificate**
   * **Type:** Categorical (string)
   * **Meaning:** Age-group / content certificate assigned to the movie.
   * **Typical values:** `U`, `A`, `UA`, `R`, `PG`, `PG-13`, `G`, `Passed`, `Approved`, `TV-PG`, `GP`.
   * **Note:** Has **101 missing values** — needs imputation.

5. **Runtime**
   * **Type:** String → Numeric (after cleaning)
   * **Meaning:** Total duration of the movie in minutes.
   * **Stored as:** `'142 min'` — must extract the numeric portion.

6. **Genre**
   * **Type:** Categorical (string, multi-label)
   * **Meaning:** Genre(s) of the movie, comma-separated.
   * **Examples:** `'Drama'`, `'Crime, Drama'`, `'Action, Crime, Drama'`.
   * **Note:** Many movies have 2–3 genres. For modeling we extract the **primary genre** (first listed).

7. **IMDB_Rating**
   * **Type:** Numeric (float) — **Target variable**
   * **Meaning:** IMDB user rating of the movie on a scale of 1.0–10.0.
   * **Range in dataset:** 7.6 – 9.3 (top 1000 movies, so all are highly rated).
   * This is what we aim to predict.

8. **Overview**
   * **Type:** Text (string)
   * **Meaning:** Short plot summary / synopsis of the movie.
   * **Note:** Useful for NLP tasks; dropped in this basic EDA pipeline.

9. **Meta_score**
   * **Type:** Numeric (float)
   * **Meaning:** Metacritic weighted average score (0–100) from professional critics.
   * **Note:** Has **157 missing values** — needs imputation.

10. **Director**
    * **Type:** Categorical (string)
    * **Meaning:** Full name of the movie's director.
    * **Top directors:** Alfred Hitchcock (14), Steven Spielberg (13), Hayao Miyazaki (11).

11. **Star1 / Star2 / Star3 / Star4**
    * **Type:** Categorical (string)
    * **Meaning:** Names of the top four billed cast members.
    * **Note:** Each is a separate column for the 1st, 2nd, 3rd, and 4th lead actor/actress.

12. **No_of_Votes**
    * **Type:** Numeric (integer)
    * **Meaning:** Total number of user ratings/votes submitted on IMDB.
    * **Range:** 25,088 – 2,343,110. Higher votes indicate more popular / mainstream titles.

13. **Gross**
    * **Type:** String → Numeric (after cleaning)
    * **Meaning:** Worldwide box-office gross revenue of the movie in USD.
    * **Stored as:** `'28,341,469'` (with commas) — must remove commas before converting.
    * **Note:** Has **169 missing values** — highest missingness in the dataset.


In [ ]:
df.head(10)


## Step 3 – Basic structure and column overview

**What:** Inspect the shape (rows, columns) and column names.

**Why:**
- To understand dataset size
- To see which features are available

**What we learn:**
- Number of samples (rows) → **1000 movies**
- Number of features (columns) → **16 columns**
- Names of the columns we will analyze


In [ ]:
# Step 3: Basic structure
print('Number of rows and columns:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())


## Step 4 – Data types and missing values

**What:**
- Use `.info()` to see data types and non-null counts per column
- Use `.isna().sum()` to count missing values per column

**Why:**
- Data types (int, float, object) tell us which operations are valid on each column
- Missing values must be handled before training any machine learning model

**Key observations to expect:**
- `Released_Year`, `Runtime`, `Gross` → stored as **strings**, need type conversion
- `Certificate` → **101 missing values**
- `Meta_score` → **157 missing values**
- `Gross` → **169 missing values** (highest)

**What we learn:**
- Which columns are numeric vs categorical
- Exactly where we have missing data


In [ ]:
# Step 4: Data types and missing values
print('--- DataFrame info ---')
print(df.info())


In [ ]:
print('\n--- Missing values per column ---')
print(df.isnull().sum())


In [ ]:
# Peek at unique values for key categorical columns
print('Certificate unique values:')
print(df['Certificate'].value_counts())


In [ ]:
# Peek at value counts for all columns
for col in df.columns:
    print(f'\n=== {col} ===')
    print(df[col].value_counts().head(10))


## Step 5 – Drop irrelevant columns and clean raw strings

**What:**
- Drop columns that contribute no predictive signal: `Poster_Link`, `Overview`, `Series_Title`
- Strip leading/trailing whitespace from all string (object) columns

**Why:**
- `Poster_Link` is a URL with no modeling value
- `Overview` is free-form text requiring NLP (out of scope for this notebook)
- `Series_Title` is a unique identifier — every row has a different title, so it adds noise not signal
- Whitespace causes silent mismatches: `' Drama'` ≠ `'Drama'`

**What we learn:**
- How to reduce the feature space to the most relevant predictors


In [ ]:
# Step 5: Drop irrelevant columns and clean strings
df_clean = df.copy()

# Drop columns not useful for ML modeling
df_clean.drop(columns=['Poster_Link', 'Overview', 'Series_Title'], inplace=True)

# Strip whitespace from all object columns
obj_cols = df_clean.select_dtypes(include=['object']).columns
for col in obj_cols:
    df_clean[col] = df_clean[col].str.strip()

print('Remaining columns:', df_clean.columns.tolist())
print('Shape:', df_clean.shape)


## Step 6 – Fix data types: `Runtime`, `Released_Year`, and `Gross`

**What:**
- `Runtime` is stored as `'142 min'` → remove `' min'` and convert to integer
- `Released_Year` is stored as a string → convert to numeric (non-numeric entries become NaN)
- `Gross` is stored as `'28,341,469'` (with commas) → remove commas and convert to float

**Why:**
- Machine learning models require numeric inputs for math operations
- Keeping these as strings prevents statistics, correlations, and scaling from working correctly

**What we learn:**
- How to parse and clean mixed-format columns in a real-world dataset


In [ ]:
# Step 6: Fix data types

# Runtime: strip ' min' and convert to integer
df_clean['Runtime'] = df_clean['Runtime'].str.replace(' min', '', regex=False).str.strip()
df_clean['Runtime'] = pd.to_numeric(df_clean['Runtime'], errors='coerce')

# Released_Year: coerce any non-numeric entries (like 'PG') to NaN
df_clean['Released_Year'] = pd.to_numeric(df_clean['Released_Year'], errors='coerce')

# Gross: remove commas, then convert to float
df_clean['Gross'] = df_clean['Gross'].str.replace(',', '', regex=False)
df_clean['Gross'] = pd.to_numeric(df_clean['Gross'], errors='coerce')

print('Updated dtypes:')
print(df_clean[['Runtime', 'Released_Year', 'Gross']].dtypes)
print()
print(df_clean[['Runtime', 'Released_Year', 'Gross']].head(5))


## Step 7 – Extract primary genre from `Genre`

**What:** The `Genre` column contains multiple genres separated by commas (e.g., `'Action, Crime, Drama'`).
We extract only the **first (primary) genre** as a new column `Primary_Genre`.

**Why:**
- Multi-label genre strings cannot be directly encoded as a single category
- The first listed genre is typically the dominant genre of the movie
- This keeps encoding simple for our baseline model

**What we learn:**
- How to parse structured string data into clean categorical features


In [ ]:
# Step 7: Extract primary genre
df_clean['Primary_Genre'] = df_clean['Genre'].str.split(',').str[0].str.strip()

print('Primary Genre distribution (top 15):')
print(df_clean['Primary_Genre'].value_counts().head(15))


## Step 8 – Check for duplicate rows

**What:** Check whether any rows are completely identical across all columns.

**Why:**
- Duplicate rows can bias both the analysis and the trained model by over-representing certain movies

**What we learn:**
- Whether we need to remove duplicate records before modeling


In [ ]:
# Step 8: Check and drop duplicate rows
num_duplicates = df_clean.duplicated().sum()
print('Number of duplicate rows:', num_duplicates)

if num_duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print('Duplicates dropped. New shape:', df_clean.shape)
else:
    print('No duplicate rows found.')


## Step 9 – Descriptive statistics

**What:**
- Use `.describe()` on numeric columns to see summary statistics
- Use `.describe(include='object')` on categorical columns to see counts and top categories

**Why:**
- To understand the range, central tendency, and spread of numeric variables
- To see which categories are most common

**What we learn:**
- `IMDB_Rating` ranges from 7.6 to 9.3 — all top-rated movies, narrow spread
- `No_of_Votes` is highly right-skewed (mean ≫ median)
- `Meta_score` ranges from 28 to 100 with mean ~77
- `Gross` is extremely skewed — a few blockbusters earn far more than the rest


In [ ]:
# Step 9: Descriptive statistics
print('--- Numeric columns summary ---')
df_clean.describe()


In [ ]:
print('\n--- Categorical columns summary ---')
df_clean.select_dtypes(include=['object']).describe()


## Step 10 – Target variable (`IMDB_Rating`) distribution

**What:**
- Plot the distribution of IMDB Ratings
- Check mean, median, and skewness

**Why:**
- To understand how ratings are distributed among the top 1000 movies
- Since this is a **regression** problem, knowing the target shape matters
- The distribution is expected to be narrow and right-skewed (all are top movies, ratings 7.6–9.3)

**What we learn:**
- Most top-1000 movies cluster between **7.7 – 8.2**
- Very few movies exceed 8.5 — those are the elite titles (Shawshank, Godfather, Dark Knight)
- Skewness ≈ **+1.02** → right-skewed distribution


In [ ]:
# Step 10: Target variable distribution
print('IMDB_Rating statistics:')
print(df_clean['IMDB_Rating'].describe())
print('\nSkewness:', round(df_clean['IMDB_Rating'].skew(), 4))

plt.figure(figsize=(9, 5))
sns.histplot(df_clean['IMDB_Rating'], bins=20, kde=True, color='steelblue', edgecolor='black')
plt.axvline(df_clean['IMDB_Rating'].mean(), color='red', linestyle='--', label=f"Mean: {df_clean['IMDB_Rating'].mean():.2f}")
plt.axvline(df_clean['IMDB_Rating'].median(), color='green', linestyle='--', label=f"Median: {df_clean['IMDB_Rating'].median():.2f}")
plt.title('Distribution of IMDB Ratings (Target Variable)')
plt.xlabel('IMDB Rating')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()


## Step 11 – Univariate EDA: Numeric feature distributions

**What:** Plot histograms for all numeric features.

**Why:**
- To understand the shape and spread of each numeric feature
- To identify skewness, outliers, or unusual patterns before modeling

**What we learn:**
- `Released_Year`: Most top-1000 movies are from the 1990s–2010s
- `Runtime`: Most movies are 90–160 minutes; a few outliers (200+ min epics)
- `Meta_score`: Roughly normal, centered around 75–80
- `No_of_Votes`: Highly right-skewed — most movies have moderate votes, a few have millions
- `Gross`: Extremely right-skewed — blockbusters dominate revenue


In [ ]:
# Step 11: Histograms for numeric features
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric columns:', numeric_cols)

df_clean[numeric_cols].hist(figsize=(16, 10), bins=25, color='steelblue', edgecolor='black')
plt.suptitle('Histograms of Numeric Features', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


## Step 12 – Univariate EDA: Categorical feature distributions

**What:** Plot bar charts for the top categories in each categorical feature.

**Why:**
- To see how many movies fall into each category (Genre, Certificate, Director, etc.)
- To identify dominant categories and rare categories that may need grouping

**What we learn:**
- **Genre**: Drama dominates the top 1000; Crime and Comedy follow
- **Certificate**: U, A, UA, R are the most common ratings
- **Director**: Alfred Hitchcock (14 movies), Steven Spielberg (13), Hayao Miyazaki (11)


In [ ]:
# Step 12: Bar charts for categorical features
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print('Categorical columns:', categorical_cols)

for col in categorical_cols:
    top_vals = df_clean[col].value_counts().head(15)
    plt.figure(figsize=(12, 5))
    sns.barplot(x=top_vals.values, y=top_vals.index, palette='Blues_r')
    plt.title(f'Top 15 values – {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## Step 13 – Bivariate EDA: Numeric features vs `IMDB_Rating`

**What:** Use scatter plots to compare each numeric feature against the target (`IMDB_Rating`).

**Why:**
- To discover linear or non-linear relationships between numeric features and the rating
- Features with visible trends are likely good predictors

**What we learn:**
- `Meta_score` → **positive correlation** with IMDB_Rating (critics and users tend to agree)
- `No_of_Votes` → higher votes weakly associated with higher rating (popular movies rated well)
- `Gross` → weak positive trend (blockbusters tend to score higher)
- `Released_Year` → slight upward trend (newer movies slightly better rated on average)
- `Runtime` → moderate positive relationship (longer movies often rated higher in top 1000)


In [ ]:
numeric_cols


In [ ]:
# Step 13: Scatter plots of numeric features vs IMDB_Rating
numeric_features = [col for col in numeric_cols if col != 'IMDB_Rating']

for col in numeric_features:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(x=df_clean[col], y=df_clean['IMDB_Rating'], alpha=0.5, color='steelblue')
    sns.regplot(x=df_clean[col], y=df_clean['IMDB_Rating'],
                scatter=False, color='red', line_kws={'linewidth': 1.5})
    plt.title(f'{col} vs IMDB_Rating')
    plt.xlabel(col)
    plt.ylabel('IMDB_Rating')
    plt.tight_layout()
    plt.show()


## Step 14 – Bivariate EDA: Categorical features vs `IMDB_Rating`

**What:** Use boxplots to show how `IMDB_Rating` varies across the top categories of each
categorical feature (Certificate, Primary_Genre, Director, Stars).

**Why:**
- To identify which categories consistently produce higher-rated movies
- To detect outliers within specific categories

**What we learn:**
- **Primary_Genre**: Biography and History genres tend to score higher; Horror slightly lower
- **Certificate**: Movies rated `A` and `Passed` tend to have slightly higher median ratings
- **Director**: Kubrick, Kurosawa, Miyazaki consistently produce 8.0+ films


In [ ]:
categorical_cols


In [ ]:
# Step 14: Boxplots of IMDB_Rating by categorical feature
key_cats = ['Certificate', 'Primary_Genre', 'Director', 'Star1']

for col in key_cats:
    top_cats = df_clean[col].value_counts().head(12).index
    subset = df_clean[df_clean[col].isin(top_cats)]
    order = subset.groupby(col)['IMDB_Rating'].median().sort_values(ascending=False).index

    plt.figure(figsize=(14, 6))
    sns.boxplot(x=col, y='IMDB_Rating', data=subset, order=order, palette='Blues')
    plt.title(f'IMDB_Rating by {col} (Top 12 categories)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## Step 15 – Correlation matrix for numeric features

**What:** Compute and visualize a Pearson correlation matrix for all numeric features.

**Why:**
- To identify pairs of features that move together (positive or negative correlation)
- To detect multicollinearity between predictors
- To see which numeric features are most correlated with the target (`IMDB_Rating`)

**What we learn:**
- `Meta_score` has the **strongest positive correlation** with `IMDB_Rating` (~0.27)
- `No_of_Votes` has a moderate positive correlation with `IMDB_Rating`
- `Gross` weakly correlates with both rating and votes
- `Released_Year` has very weak correlation with rating (era doesn't strongly predict quality)


In [ ]:
# Step 15: Correlation matrix
plt.figure(figsize=(12, 8))
corr = df_clean.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, fmt='.2f', linewidths=0.5, cmap='coolwarm', center=0)
plt.title('Correlation Matrix – Numeric Features')
plt.tight_layout()
plt.show()


## Step 16 – Handle missing values

**What:** Deal with remaining missing values in the cleaned dataset.

**Strategy used (column-specific):**
| Column | Missing | Strategy | Reason |
|---|---|---|---|
| `Certificate` | 101 | Fill with **mode** (`'U'`) | Most common certificate category |
| `Meta_score` | 157 | Fill with **median** | Robust to outliers; numeric column |
| `Gross` | 169 | Fill with **median** | Highly skewed; median more representative |
| `Released_Year` | ~1 (rare) | Fill with **median** | Numeric, very few missing |

**Why median over mean for numeric?**
- `Gross` and `No_of_Votes` are heavily right-skewed — mean is inflated by blockbusters
- Median gives a more realistic "typical" value for imputation

**What we learn:**
- How to apply column-specific imputation strategies in a real project


In [ ]:
# Step 16: Handle missing values
print('Missing values before imputation:')
print(df_clean.isnull().sum())
print()

df_model = df_clean.copy()

# Numeric columns → fill with median
num_cols = df_model.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df_model[col].isnull().sum() > 0:
        median_val = df_model[col].median()
        df_model[col] = df_model[col].fillna(median_val)
        print(f'  [Numeric]  Filled "{col}" with median = {median_val}')

# Categorical columns → fill with mode
cat_cols = df_model.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df_model[col].isnull().sum() > 0:
        mode_val = df_model[col].mode()[0]
        df_model[col] = df_model[col].fillna(mode_val)
        print(f'  [Categoric] Filled "{col}" with mode = {mode_val}')

print()
print('Missing values after imputation:')
print(df_model.isnull().sum())


## Step 17 – Encode categorical variables (Label Encoding)

**What:** Convert all categorical (string) columns into numeric form using **Label Encoding**.

Label Encoding:
- Assigns a unique integer to each category (e.g., `'Drama'` → 0, `'Crime'` → 1, ...)
- Memory-efficient and simple

**Why:**
- All ML algorithms require numeric inputs — strings cannot be fed directly
- Label encoding works well for **tree-based models** (Random Forest, XGBoost)
- For **linear models**, one-hot encoding is preferable (to avoid implied ordinal relationships)

**Note:** We drop the original `Genre` column since `Primary_Genre` captures it.

**What we learn:**
- How to transform a fully mixed dataset into a numeric matrix ready for ML


In [ ]:
# Step 17: Label encode categorical variables

# Drop original Genre since Primary_Genre replaces it
df_model.drop(columns=['Genre'], inplace=True)

categorical_cols_model = df_model.select_dtypes(include=['object']).columns.tolist()
print('Categorical columns to encode:', categorical_cols_model)

le = LabelEncoder()
df_encoded = df_model.copy()

for col in categorical_cols_model:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

print('\nShape after encoding:', df_encoded.shape)
df_encoded.head()


## Step 18 – Split into features (X) and target (y), then train/test sets

**What:**
- Separate the target column (`IMDB_Rating`) from the feature columns
- Split the data into **training** (80%) and **test** (20%) sets

**Why:**
- The model **learns** from the training set
- The test set simulates **new, unseen data** for evaluation
- An 80/20 split is standard for datasets of ~1000 rows

**What we learn:**
- Best practice for building and evaluating supervised ML models
- `train_test_split` with `random_state=42` ensures reproducible results


In [ ]:
# Step 18: Create X (features) and y (target), then split
y = df_encoded['IMDB_Rating']
X = df_encoded.drop(columns=['IMDB_Rating'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)
print()
print('y_train statistics:')
print(y_train.describe())


## Step 19 – Feature scaling (standardization)

**What:** Scale all features to have mean ≈ 0 and standard deviation ≈ 1 using `StandardScaler`.

**Why:**
- Features like `No_of_Votes` (range: 25K–2.3M) and `Gross` (range: millions) dwarf features like
  `Runtime` (90–200) and `IMDB_Rating` (7.6–9.3) — this creates bias in distance-based models
- Models that benefit from scaling: **Linear Regression, KNN, SVM, Neural Networks**
- Tree-based models (Random Forest, XGBoost) do NOT require scaling — but it doesn't hurt

**Critical rule:** Always fit the scaler **only on training data**, then apply it to test data.
Fitting on test data would cause **data leakage**.

**What we learn:**
- How to properly apply feature normalization without leaking information from the test set


In [ ]:
# Step 19: Standardize features
scaler = StandardScaler()

# Fit ONLY on training data, then transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaling complete.')
print('X_train_scaled shape:', X_train_scaled.shape)
print('X_test_scaled shape: ', X_test_scaled.shape)
print()
print('Sample scaled values (first row of X_train_scaled):')
print(X_train_scaled[0])


## Step 20 – Summary

In this notebook, we performed **end-to-end EDA and data preprocessing** for the IMDB Top 1000 Movies dataset:

1. Imported necessary Python libraries
2. Loaded the dataset (`imdb_top_1000.csv`) into a pandas DataFrame — **1000 rows × 16 columns**
3. Described all 13 features and the target variable in detail
4. Examined data types and identified key missing values:
   - `Certificate`: 101 missing | `Meta_score`: 157 missing | `Gross`: 169 missing
5. Dropped irrelevant columns (`Poster_Link`, `Overview`, `Series_Title`) and cleaned whitespace
6. Fixed mixed-type columns: extracted numeric values from `Runtime`, `Released_Year`, `Gross`
7. Extracted **Primary Genre** from the multi-label `Genre` column
8. Checked for and confirmed **no duplicate rows**
9. Generated descriptive statistics for all numeric and categorical features
10. Analyzed the target (`IMDB_Rating`) distribution — right-skewed, range 7.6–9.3, mean ≈ 7.95
11. Performed univariate EDA — histograms and bar charts for all features
12. Performed bivariate EDA — scatter plots and boxplots vs `IMDB_Rating`
13. Built a **correlation matrix** — `Meta_score` has highest correlation with `IMDB_Rating`
14. Handled missing values using **median** (numeric) and **mode** (categorical) imputation
15. Label-encoded all categorical variables
16. Split data into **80% train / 20% test** sets
17. Applied `StandardScaler` for feature normalization (no data leakage)

**From here, you can continue by:**
- Training regression models: **Linear Regression, Random Forest Regressor, XGBoost, SVR**
- Evaluating with **MAE, RMSE, and R² score**
- Analyzing **feature importance** to see what drives IMDB ratings
- Applying **NLP on `Overview`** for additional textual signal
- Exploring **star/director combinations** that consistently produce high ratings

This pipeline gives you a complete, beginner-friendly foundation for working with the IMDB movies dataset.
